# Preprocessing: Kiswahili/Somali/Dholuo PSA dataset - shared parallel test set

**What this notebook produces**, all under `data/other_langs/`:
- `other_langs_train.csv` - the training pool (English/Kiswahili/Somali/Dholuo)
- `other_langs_test.csv` - **985 rows, the same content, shared across all three
  languages** - English+Kiswahili from `Final_Training`'s own `test_psa.csv`,
  Somali+Dholuo looked up from `Filtered_PSA_final.csv` by matching English text
- `other_langs_validation.csv` - same construction, from `validation_psa.csv`

**Why a shared test set, not one per language:** evaluating each language against a
*different* set of underlying content (as an earlier version of this pipeline did) means
a score difference could just reflect one test subset being easier or harder, not a real
difference between languages. A shared set removes that confound - same content, same
difficulty, only the target language differs.

**Confirmed feasible, not assumed:** 985 of `test_psa.csv`'s 1,948 rows (50.6%) have a
matching English entry in `Filtered_PSA_final.csv` with usable Kiswahili+Somali+Dholuo -
that intersection, no other filter applied, is the resulting test set. Smaller than
`test_psa.csv` alone, but in the normal range for MT evaluation (FLORES-200's devtest is
~1,000 sentences). Kiswahili is backfilled from the same lookup only where `test_psa.csv`'s
own value is blank (confirmed a real ~40% blank rate there) - Somali/Dholuo have no other
source and always come from the lookup.

**What happens to the rows this construction doesn't use:** `Filtered_PSA_final.csv`'s
own train/validation/test split is not reused as a split at all here - every one of its
rows that isn't part of the shared test/validation set (by English-text match) becomes
training data instead, since a separate internal test/validation split is no longer
needed once the shared set covers that job for all three languages.

## 1. Setup

In [1]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")
if "final_training" not in str(PROJECT_ROOT).lower().replace(" ", "_"):
    print("\n  WARNING: 'Final_Training' doesn't appear in PROJECT_ROOT above - this "
          "notebook must live under Final_Training/Notebooks/ for find_project_root() "
          "to land in the right place. Move it there before continuing.")

from arch_config import DATA_DIR
import pandas as pd

OTHER_LANGS_DATA_DIR = Path(DATA_DIR) / "other_langs"
print(f"data dir: {Path(DATA_DIR).resolve()}")
print(f"other_langs dir: {OTHER_LANGS_DATA_DIR.resolve()}")


project root: /home/jovyan/PSA/Final_Training
data dir: /home/jovyan/PSA/Final_Training/data
other_langs dir: /home/jovyan/PSA/Final_Training/data/other_langs


## 2. Load the three source files

In [2]:
DATASET_PATH = OTHER_LANGS_DATA_DIR / "Filtered_PSA_final.csv"
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"{DATASET_PATH} not found - upload it there first.")

filtered = pd.read_csv(DATASET_PATH)
print(f"Filtered_PSA_final.csv: {len(filtered):,} rows")
print(f"columns: {list(filtered.columns)}")

test_psa_path = Path(DATA_DIR) / "test_psa.csv"
val_psa_path = Path(DATA_DIR) / "validation_psa.csv"
if not test_psa_path.exists() or not val_psa_path.exists():
    raise FileNotFoundError(
        f"{test_psa_path} and/or {val_psa_path} not found - these are Final_Training's "
        f"own files (built by 00_data_prep.ipynb), not something this notebook creates."
    )
test_psa = pd.read_csv(test_psa_path)
val_psa = pd.read_csv(val_psa_path)
print(f"\ntest_psa.csv: {len(test_psa):,} rows")
print(f"validation_psa.csv: {len(val_psa):,} rows")


Filtered_PSA_final.csv: 22,334 rows
columns: ['concept_id', 'English', 'Kiswahili', 'Somali', 'Dholuo', 'resource', 'dataset_origin', 'source_url', 'PSA_non_PSA', 'human_reviewed', 'split']

test_psa.csv: 1,948 rows
validation_psa.csv: 1,751 rows


## 3. Data quality: same-language mislabeling check

In [3]:
def flag_same_language(df, col_a, col_b):
    a = df[col_a].astype(str).str.strip().str.lower()
    b = df[col_b].astype(str).str.strip().str.lower()
    return a == b

def report_and_drop(df, mask, label):
    if mask.sum():
        print(f"{label}: dropping {mask.sum()} row(s) - identical text in both columns")
        df = df[~mask].reset_index(drop=True)
    else:
        print(f"{label}: no same-language rows found - already clean")
    return df

for target_col in ["Kiswahili", "Somali", "Dholuo"]:
    filtered = report_and_drop(filtered, flag_same_language(filtered, "English", target_col),
                               f"filtered (English/{target_col})")


filtered (English/Kiswahili): no same-language rows found - already clean
filtered (English/Somali): dropping 7 row(s) - identical text in both columns
filtered (English/Dholuo): dropping 119 row(s) - identical text in both columns


## 4. Build the shared parallel test set (985 rows) and validation set

Matched by normalized English text. A row's Somali/Dholuo always comes from this lookup
(there is no other source for either). Kiswahili is different: `test_psa.csv`/
`validation_psa.csv` already carry their own Kiswahili column, so that value is kept
when present (it's the more directly-curated source) - the lookup only **backfills**
Kiswahili where the source file's own value is blank. Confirmed directly: `test_psa.csv`'s
own Kiswahili is blank for 786 of 1,948 rows (40.3%, a real gap in that file, unrelated to
this notebook), and `Filtered_PSA_final.csv` has a usable Kiswahili value for every single
one of the rows that gap affects within the shared set - so this backfill brings Kiswahili
to the same 985/985 coverage Somali/Dholuo already have, instead of silently evaluating
Kiswahili on a smaller, non-random subset later. All three languages must have a non-blank
value (after backfill) to survive into the shared set - a row missing any one of them
doesn't work as a *shared* set.

In [4]:
def norm(s):
    return str(s).strip().lower()

filtered["_norm_en"] = filtered["English"].map(norm)
lookup = filtered.drop_duplicates(subset="_norm_en").set_index("_norm_en")

def build_shared_set(source_df, label):
    df = source_df.copy()
    df["_norm_en"] = df["English"].map(norm)
    has_match = df["_norm_en"].isin(lookup.index)
    matched = df[has_match].copy()
    matched["Somali"] = matched["_norm_en"].map(lookup["Somali"])
    matched["Dholuo"] = matched["_norm_en"].map(lookup["Dholuo"])

    # Kiswahili: keep the source file's own value where present, backfill from the same
    # lookup only where it's blank - the source file's own column is ~40% blank (a real
    # gap, confirmed against the raw file), while Filtered_PSA_final.csv has zero blank
    # Kiswahili rows, so this recovers full coverage without discarding anything already
    # curated in test_psa.csv/validation_psa.csv itself.
    kiswahili_blank = matched["Kiswahili"].isna() | (matched["Kiswahili"].astype(str).str.strip() == "")
    n_backfilled = int(kiswahili_blank.sum())
    if n_backfilled:
        matched.loc[kiswahili_blank, "Kiswahili"] = matched.loc[kiswahili_blank, "_norm_en"].map(lookup["Kiswahili"])
        print(f"{label}: backfilled Kiswahili from Filtered_PSA_final.csv for "
              f"{n_backfilled:,} row(s) whose own Kiswahili value was blank")

    all_present = (matched["Somali"].notna() & matched["Dholuo"].notna() & matched["Kiswahili"].notna() &
                   (matched["Somali"].astype(str).str.strip() != "") &
                   (matched["Dholuo"].astype(str).str.strip() != "") &
                   (matched["Kiswahili"].astype(str).str.strip() != ""))
    shared = matched[all_present].reset_index(drop=True)
    print(f"{label}: {len(shared):,} of {len(source_df):,} rows have a usable "
          f"Kiswahili+Somali+Dholuo match")
    return shared

shared_test = build_shared_set(test_psa, "shared test set (from test_psa.csv)")
print()
shared_val = build_shared_set(val_psa, "shared validation set (from validation_psa.csv)")

print("\nshared test set domain balance (psa_domain):")
print(shared_test["psa_domain"].value_counts().to_string())


shared test set (from test_psa.csv): backfilled Kiswahili from Filtered_PSA_final.csv for 703 row(s) whose own Kiswahili value was blank
shared test set (from test_psa.csv): 985 of 1,948 rows have a usable Kiswahili+Somali+Dholuo match

shared validation set (from validation_psa.csv): backfilled Kiswahili from Filtered_PSA_final.csv for 50 row(s) whose own Kiswahili value was blank
shared validation set (from validation_psa.csv): 1,589 of 1,751 rows have a usable Kiswahili+Somali+Dholuo match

shared test set domain balance (psa_domain):
psa_domain
Health         394
Agriculture    262
Education      152
Governance     111
Security        66


## 5. Build the training pool

Every `Filtered_PSA_final.csv` row that isn't part of either shared set (by English-text
match) - its own original train/validation/test split is not reused, since a separate
internal held-out set is no longer needed once the shared sets cover that job.

In [5]:
held_out_en = set(shared_test["_norm_en"]) | set(shared_val["_norm_en"])
train_pool = filtered[~filtered["_norm_en"].isin(held_out_en)].reset_index(drop=True)

print(f"Filtered_PSA_final.csv total: {len(filtered):,}")
print(f"removed (now in shared test/validation): {len(filtered) - len(train_pool):,}")
print(f"training pool: {len(train_pool):,}")


Filtered_PSA_final.csv total: 22,208
removed (now in shared test/validation): 2,870
training pool: 19,338


## 6. Save all three outputs under `data/other_langs/`

In [6]:
train_out = OTHER_LANGS_DATA_DIR / "other_langs_train.csv"
test_out = OTHER_LANGS_DATA_DIR / "other_langs_test.csv"
val_out = OTHER_LANGS_DATA_DIR / "other_langs_validation.csv"

train_pool.drop(columns=["_norm_en"]).to_csv(train_out, index=False)
shared_test.drop(columns=["_norm_en"]).to_csv(test_out, index=False)
shared_val.drop(columns=["_norm_en"]).to_csv(val_out, index=False)

print(f"saved: {train_out.resolve()}  ({len(train_pool):,} rows)")
print(f"saved: {test_out.resolve()}  ({len(shared_test):,} rows)")
print(f"saved: {val_out.resolve()}  ({len(shared_val):,} rows)")
print("\n06_nllb_other_languages.ipynb reads these same three files directly.")


saved: /home/jovyan/PSA/Final_Training/data/other_langs/other_langs_train.csv  (19,338 rows)
saved: /home/jovyan/PSA/Final_Training/data/other_langs/other_langs_test.csv  (985 rows)
saved: /home/jovyan/PSA/Final_Training/data/other_langs/other_langs_validation.csv  (1,589 rows)

06_nllb_other_languages.ipynb reads these same three files directly.
